In [1]:
from google.cloud import bigquery
from google.oauth2 import service_account

import psycopg2
import pandas as pd
from sqlalchemy import create_engine,URL

In [2]:
credentials = service_account.Credentials.from_service_account_file(
  'c:/Users/Bob/oasisbiz/datawarehouse-390004-34bcb00fb7cb.json'
)
project_id = 'datawarehouse-390004'

In [3]:
client = bigquery.Client(
  project=project_id,
  credentials=credentials
)

In [4]:
# connect to localhost
conn = psycopg2.connect(
  host='localhost',
  port=5432,
  database='postgres',
  user='postgres',
  password='postgres'
)
conn.set_session(autocommit=True)
cursor = conn.cursor()

In [5]:
engine = create_engine(
  URL.create(
    drivername='postgresql+psycopg2',
    host='localhost',
    port=5432,
    database='postgres',
    username='postgres',
    password='postgres'
  )
)

In [6]:
# get store_sales
job = client.query(
  f'''
  select
    *
  from m2.sh_bldg_sales
  where left(pnu,2) = '11'
  '''
)
store_sales_df = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


KeyboardInterrupt: 

In [ ]:
try:
  store_sales_df.to_sql(
    'store_sales',
    engine,
    if_exists='replace',
    index=False,
  )
except Exception as err:
  print(err)

In [8]:
# get market_polygon
job = client.query(
  f'''
  select
    sido_cd,
    sido_nm,
    cname,
    area,
    case
      when small = 'O' then True
      else False
    end is_small,
    case
      when medium_large = 'O' then True
      else False
    end is_medium_large,
    case
      when office = 'O' then True
      else False
    end is_office,
    case
      when collective = 'O' then True
      else False
    end is_collective,
    st_astext(geometry) geom
  from m1.polygon_area
  where sido_cd = '11'
  '''
)
market_polygon_df = job.result().to_dataframe()

c:\Users\Bob\AppData\Local\Programs\Python\Python312\Lib\site-packages\google\cloud\bigquery\table.py:1957: UserWarning: BigQuery Storage module not found, fetch data with the REST endpoint instead.
  warnings.warn(


In [9]:
try:
  cursor.execute(
    f'''
    create table market_polygon (
      sido_cd varchar,
      sido_nm varchar,
      cname varchar,
      area numeric,
      is_small boolean,
      is_medium_large boolean,
      is_office boolean,
      is_collective boolean,
      geom geometry(geometry,4326)
    )
    '''
  )
except Exception as err:
  print(err)

In [10]:
try:
  cursor.execute(
    'delete from market_polygon'
  )
  market_polygon_df.to_sql(
    'market_polygon',
    engine,
    if_exists='append',
    index=False,
  )
except Exception as err:
  print(err)